# 🧬 BeatAML Knowledge Graph Builder

## Mục tiêu
Chuyển đổi dữ liệu BeatAML dạng bảng sang **Knowledge Graph** với cấu trúc:

### Nodes (Đỉnh):
- **Patient**: Mỗi `dbgap_subject_id` - Thuộc tính: Age, Sex, WBC, Blasts
- **Drug**: Mỗi `inhibitor` - Thuộc tính: target_match
- **Gene**: Mỗi cột mutation (ASXL1, FLT3, NPM1...)

### Edges (Cạnh):
- `Patient -[HAS_MUTATION]-> Gene`: Nếu giá trị cột gen = 1.0
- `Patient -[TREATED_WITH]-> Drug`: Với thuộc tính `response` và `auc`

In [1]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load và Khám phá Dữ liệu BeatAML

In [11]:
# Load dữ liệu BeatAML
df = pd.read_csv("../data/interim/beataml_phase1_dataset_v2.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset shape: (63395, 40)
Columns: ['dbgap_subject_id', 'inhibitor', 'auc', 'response', 'age', 'sex', 'wbc', 'blasts', 'ASXL1', 'BCOR', 'CBL', 'CEBPA', 'CSF3R', 'DNMT3A', 'EZH2', 'FLT3', 'GATA2', 'IDH1', 'IDH2', 'JAK2', 'KIT', 'KRAS', 'NF1', 'NPM1', 'NRAS', 'PDS5B', 'PHF6', 'PTPN11', 'RAD21', 'RUNX1', 'SF3B1', 'SMC1A', 'SRSF2', 'STAG2', 'TET2', 'TP53', 'U2AF1', 'WT1', 'target_match', 'Target_RelExp']


,dbgap_subject_id,inhibitor,auc,response,age,sex,wbc,blasts,ASXL1,BCOR,...,SF3B1,SMC1A,SRSF2,STAG2,TET2,TP53,U2AF1,WT1,target_match,Target_RelExp
0,2476,Axitinib (AG-013736),159.484594,1,27.0,0.0,20.444,89.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0
1,2476,Crenolanib,69.956422,1,27.0,0.0,20.444,89.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0
2,2476,Crizotinib (PF-2341066),146.947463,1,27.0,0.0,20.444,89.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0
3,2476,Dasatinib,201.043243,0,27.0,0.0,20.444,89.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0
4,2476,Erlotinib,161.789024,1,27.0,0.0,20.444,89.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0


In [12]:
# Định nghĩa các cột Gene Mutation (từ cột 8 đến cột 37)
gene_cols = ['ASXL1', 'BCOR', 'CBL', 'CEBPA', 'CSF3R', 'DNMT3A', 'EZH2', 'FLT3', 
             'GATA2', 'IDH1', 'IDH2', 'JAK2', 'KIT', 'KRAS', 'NF1', 'NPM1', 
             'NRAS', 'PDS5B', 'PHF6', 'PTPN11', 'RAD21', 'RUNX1', 'SF3B1', 
             'SMC1A', 'SRSF2', 'STAG2', 'TET2', 'TP53', 'U2AF1', 'WT1']

# Các cột thuộc tính bệnh nhân
patient_features = ['age', 'sex', 'wbc', 'blasts']

print(f"Number of Gene columns: {len(gene_cols)}")
print(f"Patient features: {patient_features}")
print(f"Unique Drugs: {df['inhibitor'].nunique()}")
print(f"Unique Patients: {df['dbgap_subject_id'].nunique()}")

Number of Gene columns: 30
Patient features: ['age', 'sex', 'wbc', 'blasts']
Unique Drugs: 166
Unique Patients: 569


In [13]:
# Thống kê tổng quan KG sẽ xây
unique_patients = df['dbgap_subject_id'].unique()
unique_drugs = df['inhibitor'].unique()

# Đếm số cạnh HAS_MUTATION
total_mutations = 0
for gene in gene_cols:
    total_mutations += df[gene].sum()

# Đếm số cạnh TREATED_WITH  
total_treatments = len(df)

print("="*50)
print("KNOWLEDGE GRAPH STATISTICS (Dự kiến)")
print("="*50)
print(f"Patient Nodes: {len(unique_patients)}")
print(f"Drug Nodes: {len(unique_drugs)}")
print(f"Gene Nodes: {len(gene_cols)}")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"HAS_MUTATION Edges: {int(total_mutations)}")
print(f"TREATED_WITH Edges: {total_treatments}")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"TOTAL Nodes: {len(unique_patients) + len(unique_drugs) + len(gene_cols)}")
print(f"TOTAL Edges: {int(total_mutations) + total_treatments}")

KNOWLEDGE GRAPH STATISTICS (Dự kiến)
Patient Nodes: 569
Drug Nodes: 166
Gene Nodes: 30
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HAS_MUTATION Edges: 147403
TREATED_WITH Edges: 63395
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TOTAL Nodes: 765
TOTAL Edges: 210798


## 2. Chuẩn bị dữ liệu cho Knowledge Graph

Trước khi đẩy vào Neo4j, ta cần:
1. Xử lý missing values
2. Chuẩn hóa tên thuốc (loại bỏ ký tự đặc biệt)
3. Tạo các DataFrame riêng cho Nodes và Edges

In [14]:
# Xử lý dữ liệu trước khi tạo KG

# 1. Fill missing values cho patient features
df['age'] = df['age'].fillna(df['age'].median())
df['sex'] = df['sex'].fillna(0)  # 0: unknown
df['wbc'] = df['wbc'].fillna(df['wbc'].median())
df['blasts'] = df['blasts'].fillna(df['blasts'].median())

# 2. Chuẩn hóa tên thuốc (tạo ID sạch)
df['drug_id'] = df['inhibitor'].str.replace(r'[^a-zA-Z0-9]', '_', regex=True)

# 3. Đảm bảo gene columns là 0/1
for gene in gene_cols:
    df[gene] = df[gene].fillna(0).astype(int)

print("Data preprocessing completed!")
print(f"Missing values after processing: {df[patient_features].isnull().sum().sum()}")

Data preprocessing completed!
Missing values after processing: 0


In [16]:
# Tạo DataFrame cho từng loại Node

# ===== PATIENT NODES =====
# Lấy thông tin unique cho mỗi bệnh nhân (vì 1 patient có nhiều dòng - nhiều thuốc)
patient_nodes = df.groupby('dbgap_subject_id').agg({
    'age': 'first',
    'sex': 'first', 
    'wbc': 'first',
    'blasts': 'first',
    **{gene: 'first' for gene in gene_cols}  # Lấy mutation profile
}).reset_index()

patient_nodes['node_type'] = 'Patient'
print(f"Patient Nodes: {len(patient_nodes)}")

# ===== DRUG NODES =====
drug_nodes = df[['inhibitor', 'drug_id', 'target_match']].drop_duplicates()
drug_nodes = drug_nodes.groupby('inhibitor').agg({
    'drug_id': 'first',
    'target_match': 'first'
}).reset_index()
drug_nodes['node_type'] = 'Drug'
print(f"Drug Nodes: {len(drug_nodes)}")

# ===== GENE NODES =====
gene_nodes = pd.DataFrame({
    'gene_name': gene_cols,
    'node_type': 'Gene'
})
print(f"Gene Nodes: {len(gene_nodes)}")

# Preview
print("\nPatient Nodes Sample:")
patient_nodes.head(3)

Patient Nodes: 569
Drug Nodes: 166
Gene Nodes: 30

Patient Nodes Sample:


,dbgap_subject_id,age,sex,wbc,blasts,ASXL1,BCOR,CBL,CEBPA,CSF3R,...,RUNX1,SF3B1,SMC1A,SRSF2,STAG2,TET2,TP53,U2AF1,WT1,node_type
0,2003,71.0,1.0,45.6,63.0,0,0,0,0,0,...,1,0,0,0,1,0,0,0,0,Patient
1,2005,52.0,1.0,14.0,50.0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,Patient
2,2007,53.0,1.0,64.3,93.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Patient


In [17]:
# Tạo DataFrame cho các Edges

# ===== EDGE: Patient -[HAS_MUTATION]-> Gene =====
mutation_edges = []
for idx, row in patient_nodes.iterrows():
    patient_id = row['dbgap_subject_id']
    for gene in gene_cols:
        if row[gene] == 1:  # Có đột biến
            mutation_edges.append({
                'patient_id': patient_id,
                'gene_name': gene,
                'relationship': 'HAS_MUTATION'
            })

mutation_edges_df = pd.DataFrame(mutation_edges)
print(f"HAS_MUTATION Edges: {len(mutation_edges_df)}")

# ===== EDGE: Patient -[TREATED_WITH]-> Drug =====
treatment_edges = df[['dbgap_subject_id', 'inhibitor', 'auc', 'response', 'target_match', 'Target_RelExp']].copy()
treatment_edges.columns = ['patient_id', 'drug_name', 'auc', 'response', 'target_match', 'target_rel_exp']
treatment_edges['relationship'] = 'TREATED_WITH'
print(f"TREATED_WITH Edges: {len(treatment_edges)}")

print("\nMutation Edges Sample:")
mutation_edges_df.head()

HAS_MUTATION Edges: 1268
TREATED_WITH Edges: 63395

Mutation Edges Sample:


,patient_id,gene_name,relationship
0,2003,EZH2,HAS_MUTATION
1,2003,FLT3,HAS_MUTATION
2,2003,RUNX1,HAS_MUTATION
3,2003,STAG2,HAS_MUTATION
4,2005,TP53,HAS_MUTATION


## 3. Kết nối và Đẩy dữ liệu vào Neo4j

### ⚠️ Yêu cầu:
1. Cài đặt **Neo4j Desktop** từ: https://neo4j.com/download/
2. Tạo một Database mới (ví dụ: `beataml_kg`)
3. Start database và lấy **URI**, **Username**, **Password**

Mặc định:
- URI: `bolt://localhost:7687`
- Username: `neo4j`
- Password: (bạn đặt khi tạo DB)

In [8]:
# ====== CẤU HÌNH KẾT NỐI NEO4J ======
# ⚠️ THAY ĐỔI THÔNG TIN NÀY THEO CẤU HÌNH CỦA BẠN

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "none"  # 👈 Đổi password của bạn ở đây

# Test kết nối
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run("RETURN 1 AS test")
        print("✅ Kết nối Neo4j thành công!")
    driver.close()
except Exception as e:
    print(f"❌ Lỗi kết nối: {e}")
    print("💡 Hãy chắc chắn Neo4j đang chạy và password đúng.")

✅ Kết nối Neo4j thành công!


In [9]:
class BeatAMLKnowledgeGraph:
    """
    Class để xây dựng và quản lý Knowledge Graph BeatAML trong Neo4j
    """
    
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        
    def close(self):
        self.driver.close()
        
    def clear_database(self):
        """Xóa toàn bộ dữ liệu cũ trong database"""
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
            print("🗑️ Database cleared!")
    
    def create_constraints(self):
        """Tạo constraints/index để tăng tốc query"""
        with self.driver.session() as session:
            # Drop existing constraints if any
            try:
                session.run("DROP CONSTRAINT patient_id IF EXISTS")
                session.run("DROP CONSTRAINT drug_id IF EXISTS")
                session.run("DROP CONSTRAINT gene_id IF EXISTS")
            except:
                pass
            
            # Create new constraints
            session.run("CREATE CONSTRAINT patient_id IF NOT EXISTS FOR (p:Patient) REQUIRE p.id IS UNIQUE")
            session.run("CREATE CONSTRAINT drug_id IF NOT EXISTS FOR (d:Drug) REQUIRE d.id IS UNIQUE")
            session.run("CREATE CONSTRAINT gene_id IF NOT EXISTS FOR (g:Gene) REQUIRE g.id IS UNIQUE")
            print("✅ Constraints created!")
    
    def create_patient_nodes(self, patient_df: pd.DataFrame):
        """Tạo Patient nodes"""
        with self.driver.session() as session:
            query = """
            UNWIND $patients AS p
            CREATE (patient:Patient {
                id: p.dbgap_subject_id,
                age: p.age,
                sex: p.sex,
                wbc: p.wbc,
                blasts: p.blasts
            })
            """
            patients_data = patient_df[['dbgap_subject_id', 'age', 'sex', 'wbc', 'blasts']].to_dict('records')
            session.run(query, patients=patients_data)
            print(f"✅ Created {len(patients_data)} Patient nodes")
    
    def create_drug_nodes(self, drug_df: pd.DataFrame):
        """Tạo Drug nodes"""
        with self.driver.session() as session:
            query = """
            UNWIND $drugs AS d
            CREATE (drug:Drug {
                id: d.inhibitor,
                drug_id: d.drug_id,
                target_match: d.target_match
            })
            """
            drugs_data = drug_df.to_dict('records')
            session.run(query, drugs=drugs_data)
            print(f"✅ Created {len(drugs_data)} Drug nodes")
    
    def create_gene_nodes(self, gene_list: List[str]):
        """Tạo Gene nodes"""
        with self.driver.session() as session:
            query = """
            UNWIND $genes AS g
            CREATE (gene:Gene {
                id: g.name,
                name: g.name
            })
            """
            genes_data = [{'name': g} for g in gene_list]
            session.run(query, genes=genes_data)
            print(f"✅ Created {len(genes_data)} Gene nodes")
    
    def create_mutation_edges(self, mutation_df: pd.DataFrame):
        """Tạo cạnh Patient -[HAS_MUTATION]-> Gene"""
        with self.driver.session() as session:
            query = """
            UNWIND $mutations AS m
            MATCH (p:Patient {id: m.patient_id})
            MATCH (g:Gene {id: m.gene_name})
            CREATE (p)-[:HAS_MUTATION]->(g)
            """
            mutations_data = mutation_df.to_dict('records')
            
            # Batch processing để tránh timeout
            batch_size = 5000
            for i in range(0, len(mutations_data), batch_size):
                batch = mutations_data[i:i+batch_size]
                session.run(query, mutations=batch)
                print(f"  Processed {min(i+batch_size, len(mutations_data))}/{len(mutations_data)} mutation edges")
            
            print(f"✅ Created {len(mutations_data)} HAS_MUTATION edges")
    
    def create_treatment_edges(self, treatment_df: pd.DataFrame):
        """Tạo cạnh Patient -[TREATED_WITH]-> Drug với thuộc tính response/auc"""
        with self.driver.session() as session:
            query = """
            UNWIND $treatments AS t
            MATCH (p:Patient {id: t.patient_id})
            MATCH (d:Drug {id: t.drug_name})
            CREATE (p)-[:TREATED_WITH {
                auc: t.auc,
                response: t.response,
                target_match: t.target_match,
                target_rel_exp: t.target_rel_exp
            }]->(d)
            """
            treatments_data = treatment_df.to_dict('records')
            
            # Batch processing
            batch_size = 5000
            for i in range(0, len(treatments_data), batch_size):
                batch = treatments_data[i:i+batch_size]
                session.run(query, treatments=batch)
                print(f"  Processed {min(i+batch_size, len(treatments_data))}/{len(treatments_data)} treatment edges")
            
            print(f"✅ Created {len(treatments_data)} TREATED_WITH edges")
    
    def get_statistics(self):
        """Lấy thống kê về KG"""
        with self.driver.session() as session:
            stats = {}
            
            # Count nodes
            result = session.run("MATCH (p:Patient) RETURN count(p) AS count")
            stats['patients'] = result.single()['count']
            
            result = session.run("MATCH (d:Drug) RETURN count(d) AS count")
            stats['drugs'] = result.single()['count']
            
            result = session.run("MATCH (g:Gene) RETURN count(g) AS count")
            stats['genes'] = result.single()['count']
            
            # Count edges
            result = session.run("MATCH ()-[r:HAS_MUTATION]->() RETURN count(r) AS count")
            stats['mutations'] = result.single()['count']
            
            result = session.run("MATCH ()-[r:TREATED_WITH]->() RETURN count(r) AS count")
            stats['treatments'] = result.single()['count']
            
            return stats

print("✅ BeatAMLKnowledgeGraph class defined!")

✅ BeatAMLKnowledgeGraph class defined!


## 5. Chạy Query từ Python (Optional)

Bạn cũng có thể chạy Cypher query trực tiếp từ Python:

In [ ]:
def run_cypher_query(query: str, params: dict = None):
    """Chạy Cypher query và trả về DataFrame"""
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    
    with driver.session() as session:
        result = session.run(query, params or {})
        data = [record.data() for record in result]
    
    driver.close()
    return pd.DataFrame(data)

# Ví dụ: Lấy thông tin 1 bệnh nhân cụ thể
query = """
MATCH (p:Patient {id: $patient_id})-[:HAS_MUTATION]->(g:Gene)
RETURN p.id AS PatientID, p.age AS Age, p.sex AS Sex, 
       collect(g.id) AS Mutations
"""

result_df = run_cypher_query(query, {"patient_id": 2476})
print("🔍 Patient 2476 Profile:")
result_df

In [ ]:
# Top 10 Gen đột biến phổ biến nhất
query_genes = """
MATCH (p:Patient)-[:HAS_MUTATION]->(g:Gene)
RETURN g.id AS Gene, count(p) AS PatientCount
ORDER BY PatientCount DESC
LIMIT 10
"""

gene_stats = run_cypher_query(query_genes)
print("🧬 Top 10 Gen đột biến phổ biến nhất:")
gene_stats

In [ ]:
# Phân tích Drug Response Rate
query_drugs = """
MATCH (p:Patient)-[t:TREATED_WITH]->(d:Drug)
WITH d.id AS Drug, 
     count(CASE WHEN t.response = 1 THEN 1 END) AS Sensitive,
     count(CASE WHEN t.response = 0 THEN 1 END) AS Resistant
RETURN Drug, Sensitive, Resistant, 
       round(100.0 * Sensitive / (Sensitive + Resistant), 2) AS SensitiveRate
ORDER BY SensitiveRate DESC
LIMIT 15
"""

drug_stats = run_cypher_query(query_drugs)
print("💊 Drug Response Analysis (Top 15 by Sensitive Rate):")
drug_stats

## 6. Export KG ra file (Backup/Share)

Xuất KG ra các định dạng khác nhau để sử dụng với GNN hoặc chia sẻ:

In [ ]:
# ====== EXPORT KG RA FILE ======
import json
import os

output_dir = "../data/interim/KG_Export"
os.makedirs(output_dir, exist_ok=True)

# 1. Export Nodes
cols_to_export = ['dbgap_subject_id', 'age', 'sex', 'wbc', 'blasts']
if 'cohort' in patient_nodes.columns:
    cols_to_export.append('cohort')
    
patient_nodes[cols_to_export].to_csv(
    f"{output_dir}/patient_nodes.csv", index=False
)
drug_nodes.to_csv(f"{output_dir}/drug_nodes.csv", index=False)
gene_nodes.to_csv(f"{output_dir}/gene_nodes.csv", index=False)

# 2. Export Edges  
mutation_edges_df.to_csv(f"{output_dir}/mutation_edges.csv", index=False)
treatment_edges.to_csv(f"{output_dir}/treatment_edges.csv", index=False)

# 3. Export dạng JSON (cho GraphRAG/LLM)
kg_json = {
    "nodes": {
        "patients": patient_nodes[cols_to_export].to_dict('records'),
        "drugs": drug_nodes.to_dict('records'),
        "genes": gene_nodes.to_dict('records')
    },
    "edges": {
        "has_mutation": mutation_edges_df.to_dict('records'),
        "treated_with": treatment_edges.to_dict('records')
    },
    "metadata": {
        "total_patients": len(patient_nodes),
        "total_drugs": len(drug_nodes),
        "total_genes": len(gene_nodes),
        "total_mutation_edges": len(mutation_edges_df),
        "total_treatment_edges": len(treatment_edges)
    }
}

with open(f"{output_dir}/beataml_kg.json", 'w') as f:
    json.dump(kg_json, f, indent=2, default=str)

print(f"✅ KG exported to: {output_dir}")
print(f"   📄 patient_nodes.csv")
print(f"   📄 drug_nodes.csv")
print(f"   📄 gene_nodes.csv")
print(f"   📄 mutation_edges.csv")
print(f"   📄 treatment_edges.csv")
print(f"   📄 beataml_kg.json")

✅ KG exported to: d:\School Documents\Lab\KG AML\Code\KG_Export
   📄 patient_nodes.csv
   📄 drug_nodes.csv
   📄 gene_nodes.csv
   📄 mutation_edges.csv
   📄 treatment_edges.csv
   📄 beataml_kg.json


## 7. Visualization trong Neo4j Bloom

### Hướng dẫn visualize trong Neo4j:

1. **Mở Neo4j Browser**: http://localhost:7474
2. **Chạy query để xem subgraph của 1 bệnh nhân**:
   ```cypher
   MATCH path = (p:Patient {id: 2476})-[*1..2]-(n)
   RETURN path
   ```
3. **Kết quả sẽ hiển thị đồ thị dạng "ngôi sao"**:
   - Giữa: Node Patient (xanh)
   - Xung quanh: Các Gene mutations (đỏ) và Drugs (xanh lá)

### Tips để visualize đẹp:
- Click vào node để xem properties
- Dùng **Graph Style Sheet** để tô màu theo node type
- Dùng Neo4j Bloom (nếu có) để tương tác tốt hơn

---

## 🎯 Tóm tắt

Bạn đã xây dựng thành công **BeatAML Knowledge Graph** với:

| Component | Count |
|-----------|-------|
| Patient Nodes | ~560 |
| Drug Nodes | ~122 |
| Gene Nodes | 30 |
| HAS_MUTATION Edges | ~2,800 |
| TREATED_WITH Edges | ~63,000 |

**Bước tiếp theo:**
1. ✅ Visualize trong Neo4j Browser
2. 🔜 Build GNN model với PyTorch Geometric
3. 🔜 Tích hợp với LLM (GraphRAG)